In [ ]:
!pip -q install chromadb sentence-transformers transformers pypdf torch

import re
import torch
import chromadb
from google.colab import files
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

CHUNK_SIZE = 600
CHUNK_OVERLAP = 120
TOP_K = 5

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"

print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)

print("Loading LLM...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=-1
)

print("Loading ChromaDB...")
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(
    name="pdf_collection"
)

print("READY")


def extract_pages(file):
    reader = PdfReader(file)
    pages = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()

        if text:
            text = re.sub(r"\s+", " ", text).strip()

            if text:
                pages.append({
                    "page": i + 1,
                    "text": text
                })

    return pages


def chunk_pages(pages):
    chunks = []

    for p in pages:
        text = p["text"]
        page = p["page"]

        start = 0

        while start < len(text):
            end = start + CHUNK_SIZE
            chunk = text[start:end]

            chunks.append({
                "text": chunk,
                "page": page
            })

            start += CHUNK_SIZE - CHUNK_OVERLAP

    return chunks


def build_index(chunks):
    global collection

    try:
        chroma_client.delete_collection("pdf_collection")
    except:
        pass

    collection = chroma_client.get_or_create_collection(
        name="pdf_collection"
    )

    texts = [c["text"] for c in chunks]

    embeddings = embedder.encode(
        texts,
        normalize_embeddings=True,
        convert_to_numpy=True,
        batch_size=32
    ).tolist()

    ids = [f"chunk_{i}" for i in range(len(chunks))]
    metadatas = [{"page": c["page"]} for c in chunks]

    collection.add(
        ids=ids,
        documents=texts,
        embeddings=embeddings,
        metadatas=metadatas
    )


def retrieve(query):
    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=TOP_K
    )

    retrieved = []

    docs = results["documents"][0]
    metas = results["metadatas"][0]

    for doc, meta in zip(docs, metas):
        retrieved.append({
            "text": doc,
            "page": meta["page"]
        })

    return retrieved


def build_context(chunks):
    context = ""

    for i, c in enumerate(chunks):
        context += f"""
[Source {i+1} | Page {c['page']}]
{c['text']}

"""

    return context


def ask_llm(context, query):
    prompt = f"""
You are a strict document-based question answering assistant.

Rules:
- Answer only from the provided context.
- Do not invent information.
- If the answer is not found, say:
  "I could not find the answer in the uploaded document."
- Mention page numbers when possible.

Context:
{context}

Question:
{query}

Answer:
"""

    output = generator(
        prompt,
        max_new_tokens=300,
        temperature=0.0,
        do_sample=False,
        return_full_text=False
    )

    return output[0]["generated_text"].strip()


def upload_and_index():
    uploaded = files.upload()

    if not uploaded:
        return

    pdf_path = list(uploaded.keys())[0]

    print("Reading PDF...")
    pages = extract_pages(pdf_path)

    print("Chunking...")
    chunks = chunk_pages(pages)

    print("Building index...")
    build_index(chunks)

    print("READY")

    return chunks


def ask_loop():
    while True:
        q = input("\nAsk a question (type 'exit' to quit): ")

        if q.lower() == "exit":
            break

        retrieved = retrieve(q)
        context = build_context(retrieved)
        answer = ask_llm(context, q)

        print("\nAnswer:\n")
        print(answer)


upload_and_index()
ask_loop()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Saving PRE_THESIS_1_Internship_Report_Summer_2025_21101348_Ajmain_Islam.pdf to PRE_THESIS_1_Internship_Report_Summer_2025_21101348_Ajmain_Islam.pdf
Reading PDF...
Chunking...
Building index...
READY

Ask a question (type 'exit' to quit): what is the name of the student


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:

Student's Full Name & Signature: Ajmain Islam 21101348 i
The name of the student mentioned in the source documents is Ajmain Islam. This can be seen in the header row of both Source 1 and Source 2, where it states "Student’s Full Name & Signature: Ajmain Islam". Additionally, the text confirms that the student's full name is Ajmain Islam with their signature number 21101348.

Ask a question (type 'exit' to quit): which company did the student complete his internship


Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Answer:

The student completed his internship at [Company Name] (Page 6). The text mentions that their assistance helped them gain meaningful experience and successfully complete their assigned tasks. It also states that they received value-added technical skills, enhanced problem-solving abilities, and improved understanding of data-driven systems in an industry setting. The internship provided them with valuable technical skills, which will contribute to their professional development and offer practical insights that will be beneficial for future academic and career pursuits. The text also notes that the internship was conducted in a remote working setup, where they worked with real-world datasets, performed data preprocessing and conducted exploratory data analysis to derive meaningful insights. They also had access to basic machine learning concepts like model selection, training, and evaluation. Lastly, the internship allowed them to understand about AI, RAG, and AI engineering.